# Exercise 1 — Transportation Problem


## Task

Consider the following transportation problem. Find the **cost-minimizing shipping schedule** for cases shipped from two canneries to three warehouses.

- Demand at each warehouse must be met.
- Supply at each cannery must not be exceeded.
- Shipping costs depend on the origin-destination pair.

### Shipping costs (dollars per case)

| Cannery | New York (A) | Chicago (B) | Kansas City (C) |
|---|---:|---:|---:|
| Seattle (I) | 2.5 | 1.7 | 1.8 |
| San Diego (II) | 2.5 | 1.8 | 1.4 |

### Supply

| Cannery | Cases available |
|---|---:|
| Seattle (I) | 350 |
| San Diego (II) | 650 |
| **Total** | **1000** |

### Demand

| Warehouse | Cases demanded |
|---|---:|
| New York (A) | 300 |
| Chicago (B) | 300 |
| Kansas City (C) | 300 |
| **Total** | **900** |

Implement the following model in JuMP:

$$
\min_{x_{cw}} \quad \sum_{c\in\mathcal C}\sum_{w\in\mathcal W} t_{cw}x_{cw}
$$

subject to

$$
\sum_{w\in\mathcal W}x_{cw}\le s_c \qquad \forall c\in\mathcal C,
$$

$$
\sum_{c\in\mathcal C}x_{cw}\ge d_w \qquad \forall w\in\mathcal W,
$$

$$
x_{cw}\ge0 \qquad \forall c\in\mathcal C,\;w\in\mathcal W.
$$

Here, $\mathcal C$ is the set of canneries, $\mathcal W$ the set of warehouses, $t_{cw}$ the transportation cost, $s_c$ supply, $d_w$ demand, and $x_{cw}$ the shipped quantity.


## Reference implementation


In [ ]:
using JuMP, HiGHS

C = ["Seattle", "San Diego"]
W = ["New York", "Chicago", "Kansas City"]

ship_mat = [
    2.5 1.7 1.8;
    2.5 1.8 1.4
]

supply_vec = [350, 650]
demand_vec = [300, 300, 300]

t = JuMP.Containers.DenseAxisArray(ship_mat, C, W)
s = JuMP.Containers.DenseAxisArray(supply_vec, C)
d = JuMP.Containers.DenseAxisArray(demand_vec, W)

model = Model(HiGHS.Optimizer)
@variable(model, x[C, W] >= 0)

@objective(model, Min, sum(t[c, w] * x[c, w] for c in C, w in W))
@constraint(model, supply[c in C], sum(x[c, w] for w in W) <= s[c])
@constraint(model, demand[w in W], sum(x[c, w] for c in C) >= d[w])

optimize!(model)
objective_value(model)
value.(x)
